In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

INPUT = "/content/drive/MyDrive/research/dataset2_αβ.csv"
df = pd.read_csv(INPUT)

print("Original shape:", df.shape)

drop_cols = [
    "id_student",
    "assessment_grade",
    "exam_grade",
    "VLE_normalized"
]

df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

print("After cleaning:", df_clean.shape)

OUT = "/content/drive/MyDrive/research/dataset_tabddpm_ready.csv"
df_clean.to_csv(OUT, index=False)

print("Saved:", OUT)

Original shape: (24615, 48)
After cleaning: (24615, 44)
Saved: /content/drive/MyDrive/research/dataset_tabddpm_ready.csv


In [ ]:
import pandas as pd
import numpy as np
import os


# LOAD CLEAN DATASET

INPUT = "/content/drive/MyDrive/research/dataset_tabddpm_ready.csv"
df = pd.read_csv(INPUT)

df=df.rename(columns={"final_result": "y"})

print("Loaded shape:", df.shape)


# SELECT TEMPORAL COLUMNS ONLY

temporal_cols = [c for c in df.columns if c.startswith("V_")]

print("Temporal columns:", temporal_cols)
print("Number of temporal columns:", len(temporal_cols))


# MCAR FUNCTION

def apply_mcar(df, cols, missing_rate, seed):
    df_copy = df.copy()
    np.random.seed(seed)

    n_rows = df_copy.shape[0]
    n_cols = len(cols)

    total_entries = n_rows * n_cols
    n_missing = int(total_entries * missing_rate)

    # choose unique positions (no repetition)
    indices = np.random.choice(total_entries, n_missing, replace=False)

    rows = indices // n_cols
    cols_idx = indices % n_cols

    for r, c in zip(rows, cols_idx):
        df_copy.at[r, cols[c]] = np.nan

    return df_copy


# PARAMETERS

missing_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
n_trials = 5

SAVE_DIR = "/content/drive/MyDrive/research/MCAR_DATASETS/"
os.makedirs(SAVE_DIR, exist_ok=True)


# GENERATE MCAR DATASETS

for rate in missing_rates:
    for trial in range(n_trials):

        seed = 42 + trial

        df_mcar = apply_mcar(df, temporal_cols, rate, seed)

        file_name = f"dataset_mcar_{int(rate*100)}_trial{trial+1}.csv"
        out_path = os.path.join(SAVE_DIR, file_name)

        df_mcar.to_csv(out_path, index=False)

        print(f"Saved: {file_name}")

print("✅ All MCAR datasets generated!")

Loaded shape: (24615, 44)
Temporal columns: ['V_1', 'V_2', 'V_3', 'V_4', 'V_5', 'V_6', 'V_7', 'V_8', 'V_9', 'V_10']
Number of temporal columns: 10
Saved: dataset_mcar_10_trial1.csv
Saved: dataset_mcar_10_trial2.csv
Saved: dataset_mcar_10_trial3.csv
Saved: dataset_mcar_10_trial4.csv
Saved: dataset_mcar_10_trial5.csv
Saved: dataset_mcar_20_trial1.csv
Saved: dataset_mcar_20_trial2.csv
Saved: dataset_mcar_20_trial3.csv
Saved: dataset_mcar_20_trial4.csv
Saved: dataset_mcar_20_trial5.csv
Saved: dataset_mcar_30_trial1.csv
Saved: dataset_mcar_30_trial2.csv
Saved: dataset_mcar_30_trial3.csv
Saved: dataset_mcar_30_trial4.csv
Saved: dataset_mcar_30_trial5.csv
Saved: dataset_mcar_40_trial1.csv
Saved: dataset_mcar_40_trial2.csv
Saved: dataset_mcar_40_trial3.csv
Saved: dataset_mcar_40_trial4.csv
Saved: dataset_mcar_40_trial5.csv
Saved: dataset_mcar_50_trial1.csv
Saved: dataset_mcar_50_trial2.csv
Saved: dataset_mcar_50_trial3.csv
Saved: dataset_mcar_50_trial4.csv
Saved: dataset_mcar_50_trial5.csv
✅ A

In [ ]:
import pandas as pd
import os

# PATHS
INPUT = "/content/drive/MyDrive/research/dataset2_αβ.csv"
SAVE_DIR = "/content/drive/MyDrive/research/MCAR_DATASETS"
OUTPUT = os.path.join(SAVE_DIR, "oula_complete.csv")

# COLUMNS TO DROP
drop_cols = [
    "id_student",
    "assessment_grade",
    "exam_grade",
    "VLE_normalized"
]

# LOAD
df = pd.read_csv(INPUT)
print("Original shape:", df.shape)

# DROP EXTRA COLUMNS
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# RENAME TARGET
if "final_result" in df.columns:
    df = df.rename(columns={"final_result": "y"})

# CHECK TARGET
if "y" not in df.columns:
    raise ValueError("Column 'y' not found. Also could not find 'final_result' to rename.")

print("After cleaning:", df.shape)
print("Columns:")
print(df.columns.tolist())

# SAVE
os.makedirs(SAVE_DIR, exist_ok=True)
df.to_csv(OUTPUT, index=False)

print("Saved cleaned original dataset to:", OUTPUT)

Original shape: (24615, 48)
After cleaning: (24615, 44)
Columns:
['code_module', 'code_presentation', 'gender', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'y', 'course_category', 'region_East Anglian Region', 'region_East Midlands Region', 'region_Ireland', 'region_London Region', 'region_North Region', 'region_North Western Region', 'region_Scotland', 'region_South East Region', 'region_South Region', 'region_South West Region', 'region_Wales', 'region_West Midlands Region', 'region_Yorkshire Region', 'W_1', 'V_1', 'W_2', 'V_2', 'W_3', 'V_3', 'W_4', 'V_4', 'W_5', 'V_5', 'W_6', 'V_6', 'W_7', 'V_7', 'W_8', 'V_8', 'W_9', 'V_9', 'W_10', 'V_10']
Saved cleaned original dataset to: /content/drive/MyDrive/research/MCAR_DATASETS/oula_complete.csv
